In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Locate project directory
candidates = [Path.cwd(), Path.cwd().parent]

project_root = next(
    (p for p in candidates if (p / "data" / "raw").exists()),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not locate data/raw folder")

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

# Load events with appropriate data types
events = pd.read_csv(
    raw_dir / "events.csv",
    dtype={
        "timestamp": "int64",
        "visitorid": "int64",
        "event": "category",
        "itemid": "int64",
        "transactionid": "Int64"
    }
)

print("Rows loaded:", len(events))
print("Columns:", events.columns.tolist())

# Exact duplicate audit
exact_duplicates = events.duplicated().sum()
duplicate_group_rows = events.duplicated(keep=False).sum()

print("Exact duplicate records to remove:", exact_duplicates)
print("Records belonging to duplicate groups:", duplicate_group_rows)

# Duplicate records by event type
duplicate_event_summary = (
    events.loc[events.duplicated(keep=False)]
    .groupby("event", observed=True)
    .size()
    .reset_index(name="duplicate_group_rows")
)

display(duplicate_event_summary)

# Check event types
print("\nEvent types:")
display(events["event"].value_counts())

# Check required columns for nulls
print("\nMissing values:")
display(events.isna().sum())

Rows loaded: 2756101
Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']
Exact duplicate records to remove: 460
Records belonging to duplicate groups: 918


,event,duplicate_group_rows
0,addtocart,732
1,view,186



Event types:


event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64


Missing values:


timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

In [2]:
# Remove exact duplicate records
events_cleaned = events.drop_duplicates(keep="first").copy()

# Validate
print("Original rows:", len(events))
print("Cleaned rows:", len(events_cleaned))
print("Rows removed:", len(events) - len(events_cleaned))
print("Remaining exact duplicates:", events_cleaned.duplicated().sum())

print("\nEvent counts after cleaning:")
display(events_cleaned["event"].value_counts())

print("\nMissing values after cleaning:")
display(events_cleaned.isna().sum())

Original rows: 2756101
Cleaned rows: 2755641
Rows removed: 460
Remaining exact duplicates: 0

Event counts after cleaning:


event
view           2664218
addtocart        68966
transaction      22457
Name: count, dtype: int64


Missing values after cleaning:


timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733184
dtype: int64

In [3]:
events_cleaned.to_csv(
    processed_dir / "events_cleaned.csv",
    index=False
)

print("Cleaned events dataset saved successfully.")

Cleaned events dataset saved successfully.


In [4]:
item_files = [
    raw_dir / "item_properties_part1.csv",
    raw_dir / "item_properties_part2.csv"
]

for file in item_files:
    print(f"\nAuditing: {file.name}")

    total_rows = 0
    missing_counts = None
    min_timestamp = None
    max_timestamp = None
    properties = set()

    for chunk in pd.read_csv(file, chunksize=500_000):
        total_rows += len(chunk)

        if missing_counts is None:
            missing_counts = chunk.isna().sum()
        else:
            missing_counts += chunk.isna().sum()

        chunk_min = chunk["timestamp"].min()
        chunk_max = chunk["timestamp"].max()

        min_timestamp = chunk_min if min_timestamp is None else min(min_timestamp, chunk_min)
        max_timestamp = chunk_max if max_timestamp is None else max(max_timestamp, chunk_max)

        properties.update(chunk["property"].dropna().unique())

    print("Rows:", total_rows)
    print("Missing values:")
    print(missing_counts)
    print("Unique properties:", len(properties))
    print("Timestamp range:")
    print(pd.to_datetime(min_timestamp, unit="ms", utc=True))
    print(pd.to_datetime(max_timestamp, unit="ms", utc=True))


Auditing: item_properties_part1.csv
Rows: 10999999
Missing values:
timestamp    0
itemid       0
property     0
value        0
dtype: int64
Unique properties: 1097
Timestamp range:
2015-05-10 03:00:00+00:00
2015-09-13 03:00:00+00:00

Auditing: item_properties_part2.csv
Rows: 9275903
Missing values:
timestamp    0
itemid       0
property     0
value        0
dtype: int64
Unique properties: 1094
Timestamp range:
2015-05-10 03:00:00+00:00
2015-09-13 03:00:00+00:00


In [5]:
category_tree = pd.read_csv(raw_dir / "category_tree.csv")

print("Rows:", len(category_tree))
print("\nColumns:", category_tree.columns.tolist())
print("\nData types:")
print(category_tree.dtypes)

print("\nMissing values:")
print(category_tree.isna().sum())

print("\nExact duplicate rows:", category_tree.duplicated().sum())
print("Duplicate category IDs:", category_tree["categoryid"].duplicated().sum())

print("\nUnique categories:", category_tree["categoryid"].nunique())
print("Unique parent categories:", category_tree["parentid"].nunique())

print("\nParent IDs not present as category IDs:")
invalid_parents = category_tree.loc[
    category_tree["parentid"].notna()
    & ~category_tree["parentid"].isin(category_tree["categoryid"])
]
print(len(invalid_parents))
display(invalid_parents.head(10))

Rows: 1669

Columns: ['categoryid', 'parentid']

Data types:
categoryid      int64
parentid      float64
dtype: object

Missing values:
categoryid     0
parentid      25
dtype: int64

Exact duplicate rows: 0
Duplicate category IDs: 0

Unique categories: 1669
Unique parent categories: 362

Parent IDs not present as category IDs:
0


,categoryid,parentid
